In [1]:

import pandas as pd
import re

from nltk.stem import PorterStemmer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score
)

# Step 1: Load dataset
data = pd.read_csv("spam.csv", encoding="latin1")

# Step 2: Select required columns
data = data[['v1', 'v2']]
data = data.dropna()

# Rename columns
data.columns = ['labels', 'message']

# Step 3: Text preprocessing
ps = PorterStemmer()

def preprocess(text):
    words = re.findall(r'\b[a-zA-Z]+\b', str(text).lower())
    words = [ps.stem(word) for word in words]
    return " ".join(words)

data['message'] = data['message'].apply(preprocess)

# Step 4: Convert text into numerical features
vectorizer = TfidfVectorizer(stop_words='english')

X = vectorizer.fit_transform(data['message'])

# Convert labels into numbers
y = data['labels'].map({'ham': 0, 'spam': 1})

# Step 5: Split dataset into training and testing
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Step 6: Train KNN model
model = KNeighborsClassifier(
    n_neighbors=5,
    metric='cosine',
    algorithm='brute'
)

model.fit(X_train, y_train)

# Step 7: Predict test data
y_pred = model.predict(X_test)

# Step 8: Calculate performance
print("Accuracy:", accuracy_score(y_test, y_pred))

print("Precision:", precision_score(
    y_test, y_pred, zero_division=0
))

print("Sensitivity:", recall_score(
    y_test, y_pred, zero_division=0
))

# Step 9: Confusion Matrix
cm = confusion_matrix(y_test, y_pred, labels=[0, 1])

tn, fp, fn, tp = cm.ravel()

specificity = tn / (tn + fp) if (tn + fp) != 0 else 0

print("\nConfusion Matrix:")
print(cm)

print("Specificity:", specificity)

# Step 10: Test a new message
email = ["Congratulations! You won a free lottery prize"]

email = [preprocess(email[0])]
email_vector = vectorizer.transform(email)

prediction = model.predict(email_vector)

if prediction[0] == 1:
    print("\nEmail is Spam")
else:
    print("\nEmail is Ham (Not Spam)")

Accuracy: 0.9668161434977578
Precision: 0.9375
Sensitivity: 0.8053691275167785

Confusion Matrix:
[[958   8]
 [ 29 120]]
Specificity: 0.9917184265010351

Email is Spam
